<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c41-header.png" alt="Nextia Learning · Speech AI: Transcribe, Understand and Speak" width="100%">

# Text-to-speech for prompts

**[Text-to-speech for prompts](https://learning.nextia-ai.com/courses/speech/m07/text-to-speech-for-prompts/)** · Speech AI: Transcribe, Understand and Speak · Module 7, lesson 2 of 6 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** compare two text-to-speech voices on the IVR's prompts by listening and by numbers, measure how long each prompt keeps a caller waiting, and write shorter prompts with the option before the key.

| | |
|---|---|
| **Time** | About 20 minutes |
| **Needs** | An internet connection for the setup cells, and sound (headphones are best). No account, no key and no model download. |
| **You should know** | The IVR's states and prompts, from [What an IVR is, and why companies still use one](https://learning.nextia-ai.com/courses/speech/m07/what-an-ivr-is/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, SciPy 1.18.1, soundfile 0.14.0 and matplotlib 3.11.2 (scikit-learn 1.9.1, jiwer 4.0.0 and faster-whisper 1.2.1 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/speech/m07/text-to-speech-for-prompts/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/M07-L02-text-to-speech-for-prompts/text-to-speech-for-prompts-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, scipy 1.18.1, soundfile 0.14.0, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 scipy==1.18.1 soundfile==0.14.0 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'scipy', 'soundfile']:
    print(name, version(name))

## Setup: get the Larkfield calls

Run the next cell. It downloads the files that this lesson needs (about 2 MB) into a folder `larkfield-calls`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-calls` (in Colab: the **Files** panel on the left), then run the cell again.

Where the calls come from: Larkfield is an invented shop. Our code wrote the scripts, and open text-to-speech voices (Piper and Kokoro) read them. Every call is **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/data/dataset-card.md) names each voice and its licence. The calls are free to reuse under CC BY 4.0.

In [ ]:
# Setup: download the files into larkfield-calls/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/"
FILES = {
    "data/tts/piper-en-1.wav": "bd42aa597d53f74d5c3cc374bb25786740da9e544d6d15550c00253145cd6508",
    "data/tts/piper-en-2.wav": "54ff63686698d89f2dda24083861b1249a1a4e7eb35b5433ffda4b9138504011",
    "data/tts/piper-en-3.wav": "cac9cc704a29344e2888435c0c76fa56e860102a3a900faffbdf566531b4382b",
    "data/tts/piper-en-4.wav": "699f8f54f7df8fdebc7caabe5bca5db84d12626c2d41bd88778506fd86ecce29",
    "data/tts/piper-en-5.wav": "df19219173c65cfa1f030fc26c2fdd3bcb22c4b065d1d12bde757d393934125d",
    "data/tts/piper-en-6.wav": "ac683795ffb66517cee1053e4602b98a1031c813cb162802a615e50563d46618",
    "data/tts/piper-fr-1.wav": "62958f50b04a9075322b05b90ee14f485e43c3a13398b3e4f0d1d49c3834c1a7",
    "data/tts/piper-fr-2.wav": "4102641ed94fa241e14829b9241111fe86325095743e741abafb160cb493372d",
    "data/tts/piper-fr-3.wav": "a48fab46ee17c12c753761cae1c4d4ab6844b90b77b88075a4cc955ecf966160",
    "data/tts/piper-fr-4.wav": "fe30702545f821eca1390de7ca30c06dedde81de8aff0f677508d38dcbf1687e",
    "data/tts/piper-fr-5.wav": "2655eb978b629aaae7ff8310941b232beff3c7457656c83b16c770504c60ddfb",
    "data/tts/kokoro-en-1.wav": "aaa64092d42827327c3c1a4f5a5e44b5c09133a6f29fc251d1380b419cc64efd",
    "data/tts/kokoro-en-2.wav": "0fda54560e287cfac8fbe6f852f5c651d33d94c75bb176e9d751ea7b4507c648",
    "data/tts/kokoro-en-3.wav": "9832f6ec3a88fd8fd2a66e24fff9ea2b058e2c52a751825f64c97f17ea254ac4",
    "data/tts/kokoro-en-4.wav": "36553daec634c190f216668986bf1d06654f732ad6f1bb3531a34d87ce358008",
    "data/tts/kokoro-en-5.wav": "a202e71391ba4a5b93d5adbd370c5821e3b859777c4c817df9140866cd794d80",
    "data/tts/kokoro-en-6.wav": "a347aeca85eaa649ed2e2186cd2ad99a04534241a303ae564621a3cbca4106f9",
    "data/tts/kokoro-fr-1.wav": "e3ab1a1bae826837546c607b2c937209c882400e43f8d4c018d8431846cec0a3",
    "data/tts/kokoro-fr-2.wav": "219e5552968f37aa0b15689656bc6b1def2efe7fe3ab804fdcb2f54dbd4d18cb",
    "data/tts/kokoro-fr-3.wav": "6409f110ec423ab65bac932357d44ffccef19e9b30a548174d5252643784555f",
    "data/tts/kokoro-fr-4.wav": "e2a28b3c14d538a3b21dfd9ae52164971ce3962e8e2dfbf98ff8356ddf6576df",
    "data/tts/kokoro-fr-5.wav": "340f98360679f24c1d4283fb3983e00f748efb6954a53c00d76ee63ecd5328cd",
    "compare.py": "db394eb044e0423706644ba7c8e6451e4d21a268293b868032918ea2f30db85b",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-calls").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-calls and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-calls.")

## Setup: helpers

The next cell loads NumPy as `np`, soundfile as `sf`, SciPy's `signal`, matplotlib, the audio player `Audio`, and the course helper as `cp` (normalisation, word error rate and the comparison table). It defines a few small helpers:

- `load_calls()` reads the labels of every call; `reference(call)` joins its turns into the reference transcript.
- `read_audio(path, sr, channel)` reads a recording as numbers between -1 and 1; `dbfs(x)` gives its loudness.
- `true_turns(call)` gives the true (start, end, speaker) of each turn, from the first word to the last word.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: calls, audio, and checks that do not show the answer.
import hashlib, json, warnings
from math import gcd
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import Audio, display
import compare as cp
warnings.filterwarnings("ignore")

def load_calls(path="data/calls/calls.jsonl"):
    """The labels of every call, by call ID."""
    rows = [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]
    return {c["call_id"]: c for c in rows}

def reference(call, speaker=None):
    """The reference transcript: the text of all turns (or of one speaker's turns), in time order."""
    return " ".join(t["text"] for t in call["turns"] if speaker in (None, t["speaker"]))

def read_audio(path, sr=None, channel=None):
    """(numbers between -1 and 1, samples a second). channel=None mixes the channels; 0 or 1 keeps one side."""
    x, file_sr = sf.read(str(path), dtype="float32", always_2d=True)
    x = x.mean(axis=1) if channel is None else x[:, channel]
    if sr and sr != file_sr:
        g = gcd(file_sr, sr)
        x = signal.resample_poly(x, sr // g, file_sr // g).astype(np.float32)
    return x, sr or file_sr

def dbfs(x):
    """Loudness in decibels below full scale: 0 is the loudest; speech on a phone is often -30 to -15."""
    return float(20 * np.log10(np.sqrt(np.mean(np.square(x))) + 1e-9))

def true_turns(call):
    """The true turns: (start, end, speaker), from the first word to the last word of each turn."""
    return [(t["words"][0]["start"], t["words"][-1]["end"], t["speaker"]) if t.get("words")
            else (t["start"], t["end"], t["speaker"]) for t in call["turns"]]

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Listen to the two voices

The course made the IVR's prompts with **Piper** (English voice learned from LJ Speech, French from SIWIS) and **Kokoro** (voices af_heart and ff_siwis), then sent them through Larkfield's phone line. Every voice is synthetic. Run the next cell and listen to the same confirmation prompt in both voices.

In [ ]:
PROMPTS = {
    "en": [
        "How can I help? Say order status, opening hours, or start a return. You can also press 1 for orders, 2 for opening hours, 3 for returns, or 0 for a person.",
        "Please say or type your six-digit order number.",
        "Order 7 4 2, 1 5 5. Is that right? Say yes or no, or press 1 for yes and 2 for no.",
        "I am putting you through to a person now. They will see what you told me.",
        "Your Grow Hub shipped on May 4 and is on its way.",
        "Welcome to Larkfield. You are talking to an automated assistant. For English, say English or press 1. Pour le français, dites français ou appuyez sur 2."
    ],
    "fr": [
        "Comment puis-je vous aider? Dites suivi de commande, heures d'ouverture, ou faire un retour. Vous pouvez aussi appuyer sur 1 pour les commandes, 2 pour les heures, 3 pour les retours, ou 0 pour parler à quelqu'un.",
        "Dites ou tapez votre numéro de commande à six chiffres.",
        "Commande 7 4 2, 1 5 5. Est-ce exact? Dites oui ou non, ou appuyez sur 1 pour oui et 2 pour non.",
        "Je vous transfère à une personne. Elle verra ce que vous m'avez dit.",
        "Votre Grow Hub est parti le 4 mai et il est en route."
    ]
}

clips = []
for engine in ("piper", "kokoro"):
    for lang in ("en", "fr"):
        for i, text in enumerate(PROMPTS[lang]):
            x, sr = read_audio(f"data/tts/{engine}-{lang}-{i + 1}.wav")
            clips.append({"engine": engine, "language": lang, "n": i + 1, "text": text, "seconds": len(x) / sr,
                          "words": len(text.split()), "rate": sr})
print(len(clips), "prompt recordings, all at", {c["rate"] for c in clips}, "samples a second (the phone line)")
x, sr = read_audio("data/tts/piper-en-3.wav")
display(Audio(x, rate=sr))
x, sr = read_audio("data/tts/kokoro-en-3.wav")
display(Audio(x, rate=sr))
print(PROMPTS["en"][2])

> **Check.** 22 recordings, all at 8,000 samples a second, as a caller hears them. Which voice would you rather hear on the phone? Write down why.

## 2. How long does each prompt take?

In [ ]:
print(f'{"prompt":>6} {"words":>5} {"Piper en":>9} {"Kokoro en":>10}')
for i, text in enumerate(PROMPTS["en"]):
    p = next(c for c in clips if c["engine"] == "piper" and c["language"] == "en" and c["n"] == i + 1)
    k = next(c for c in clips if c["engine"] == "kokoro" and c["language"] == "en" and c["n"] == i + 1)
    print(f'{i + 1:>6} {p["words"]:>5} {p["seconds"]:>8.1f}s {k["seconds"]:>9.1f}s   {text[:50]}')

> **Check.** The menu (prompt 1) takes 10.9 seconds with Piper, and the welcome (prompt 6) 11.0 seconds. A caller who wants the opening hours hears both before they can answer.

## 3. How fast does each voice speak?

> **Predict.** Which voice says more words a second: Piper or Kokoro?

In [ ]:
def words_per_second(engine, language):
    cs = [c for c in clips if c["engine"] == engine and c["language"] == language]
    return sum(c["words"] for c in cs) / sum(c["seconds"] for c in cs)

rates = {(e, l): round(words_per_second(e, l), 2) for e in ("piper", "kokoro") for l in ("en", "fr")}
for (e, l), r in rates.items():
    print(f"{e:7} {l}: {r} words a second")

> **Check.** Piper: 2.96 words a second in English, 3.36 in French. Kokoro: 2.91 and 3.16. Speaking speed is not the same as compute speed: the lesson's chart shows that Piper also **makes** its audio about five times faster.

## 4. Your turn: a shorter menu

> **Your turn.** Write a shorter English menu in `my_menu`: it must name orders, hours and returns, give the keys 1, 2, 3 and 0, and have at most 18 words. The cell estimates its length with Piper's speed.

In [ ]:
my_menu = ...  # your shorter English menu: every option, the keys 1, 2, 3 and 0, at most 18 words

words = None if my_menu is ... else len(my_menu.split())
if words:
    print(words, "words; about", round(words / rates[("piper", "en")], 1), "seconds with Piper (the menu now:",
          round(len(PROMPTS["en"][0].split()) / rates[("piper", "en")], 1), "seconds)")

In [ ]:
done = words is not None
expect("words in your menu, at most 18", (words <= 18) if done else None, True)
expect("it names orders, hours and returns",
       all(w in my_menu.lower() for w in ("order", "hour", "return")) if done else None, True)
expect("it gives the keys 1, 2, 3 and 0", all(k in my_menu for k in "1230") if done else None, True)

> **Check.** The solution's menu has 16 words. With Piper, it is about 6.5 seconds instead of about 11. In the lesson, you make it with Piper for real.

## 5. Your turn: the option before the key

> **Your turn.** Write `option_first(options)`. It gets pairs (option, key) and returns one sentence per pair: "For orders, press 1." Join the sentences with one space.

In [ ]:
def option_first(options):
    ...  # your code: [("orders", "1"), ("opening hours", "2")] -> "For orders, press 1. For opening hours, press 2."

print(option_first([("orders", "1"), ("opening hours", "2"), ("returns", "3"), ("a person", "0")]))

In [ ]:
menu_text = option_first([("orders", "1"), ("opening hours", "2"), ("returns", "3"), ("a person", "0")])

In [ ]:
expect_hash('the menu with the option first', menu_text, '65008a7ebc51265c')

## 6. Failure case: one voice for two languages

Listen to the welcome, the first thing every caller hears.

In [ ]:
x, sr = read_audio("data/tts/piper-en-6.wav")
display(Audio(x, rate=sr))
print(PROMPTS["en"][5])

> **Check.** The welcome has a French sentence, "Pour le français, dites français ou appuyez sur 2", but the IVR says the whole welcome with its **English** voice. A French caller hears French with an American accent, at the very start of the call. **Fix:** make each language's sentence with that language's voice, and join the two audio pieces. Listen to every prompt, in every language, before callers do.

## Next

You compared two voices by ear and by numbers, measured how long prompts keep a caller waiting, and wrote shorter ones. The [lesson page](https://learning.nextia-ai.com/courses/speech/m07/text-to-speech-for-prompts/) has the speed and intelligibility chart, the licences of each voice, and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [A keypad menu first](https://learning.nextia-ai.com/courses/speech/m07/a-keypad-menu-first/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/speech/m07/text-to-speech-for-prompts/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). Every call is synthetic: open text-to-speech voices read scripts that we invented. No real customer and no real person's voice is in this data.